# 06 — Manuscript figures 2–7

Produces the six data figures from the derived data. Output written to `outputs/figures/`.

In [ ]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))

population_dir = REPO_ROOT / 'derived_data' / 'population'
gallery_dir    = REPO_ROOT / 'derived_data' / 'm1_cells'
review_dir     = REPO_ROOT / 'derived_data' / 'review_judgements'
FIG_DIR        = REPO_ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import pdist, squareform

REPO_ROOT = REPO_ROOT
import figstyle as fs
from rf_analysis.sparse_noise import _pixel_size

fs.apply()

outputs_dir = REPO_ROOT / 'outputs'
population_dir = REPO_ROOT / 'derived_data' / 'population'
gallery_dir = REPO_ROOT / 'derived_data' / 'm1_cells'

FIG_DIR = REPO_ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = gallery_dir / 'm1_neuron_dataset.pkl'
GABOR_CSV    = gallery_dir / 'gabor_vs_gd_m1_31_v3.csv'
PIXEL_SIZE_DEG = _pixel_size('locally_sparse_noise_4deg')
SIGMA_SMOOTH   = 0.75 * PIXEL_SIZE_DEG

In [ ]:
with open(DATASET_PATH, 'rb') as f:
    dataset = pickle.load(f)

m1 = pd.DataFrame([{k: r.get(k) for k in
                    ['name', 'cell_id', 'container_id', 'sigma_deg', 'kappa',
                     'kappa_dir', 'sigma_phi_deg', 'sigma_orth_deg', 'phi_deg',
                     'theta_envelope', 'r_squared', 'x0_deg', 'y0_deg',
                     'cortex_x_um', 'cortex_y_um', 'cre_line']}
                   for r in dataset])
m1['sigma_min'] = m1[['sigma_phi_deg', 'sigma_orth_deg']].min(axis=1)

In [ ]:
review_dir = REPO_ROOT / 'derived_data' / 'review_judgements'

csvs = sorted(population_dir.glob('rf_params_order_v2_container_*.csv'))
allp = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
for col in ['sigma','theta','kappa','r_squared','sigma_x','sigma_y','x0','y0',
            'phi','cortex_x_um','cortex_y_um','delta_r2_vs_m0']:
    if col in allp.columns:
        allp[col] = pd.to_numeric(allp[col], errors='coerce')
allp['sigma_major'] = allp[['sigma_x','sigma_y']].max(axis=1)
allp['sigma_minor'] = allp[['sigma_x','sigma_y']].min(axis=1)

judged, nb07_m1 = {}, set()
pth = review_dir / 'manual_m_judgements.csv'
if pth.exists():
    j = pd.read_csv(pth); j = j[j['m_manual'] >= 0]
    for _, r in j.iterrows():
        judged[int(r['cell_id'])] = int(r['m_manual'])
        if int(r['m_manual']) == 1: nb07_m1.add(int(r['cell_id']))
for path, col in [(review_dir/'nb09_review_judgements.csv','m_manual'),
                  (population_dir/'targeted_review_judgements.csv','m_manual'),
                  (population_dir/'unreviewed_m1_judgements.csv','m_manual'),
                  (population_dir/'rescue_review_judgements.csv','m_final')]:
    if not path.exists(): continue
    j = pd.read_csv(path); j = j[j[col] >= 0]
    for _, r in j.iterrows():
        cid, mnew = int(r['cell_id']), int(r[col])
        if cid in nb07_m1 and mnew == 0: continue
        judged[cid] = mnew
allp['m_manual'] = allp['cell_id'].map(judged)
allp['manually_verified'] = allp['cell_id'].isin(judged)
allp['m_final'] = np.where(allp['m_manual'].notna(), allp['m_manual'],
                           allp['derivative_order']).astype(int)
demote = ((allp['m_final']==2) & (~allp['manually_verified']) &
          (allp['delta_r2_vs_m0']<0.10))
allp.loc[demote,'m_final'] = 0

POP = allp
N_POP = len(POP)

In [ ]:
def signed_map_axes(ax, M, title=None, vlim=None):
    M = np.asarray(M, float)
    v = vlim or np.abs(M).max()
    ax.imshow(M, cmap=fs.RF_CMAP, vmin=-v, vmax=v, origin='upper',
              interpolation='nearest', aspect='auto')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_linewidth(0.6); s.set_color(fs.GREY)
    if title:
        ax.set_title(title, fontsize=fs.FS_ANNOT, pad=3)

def model_contours(ax, M):
    M = np.asarray(M, float); peak = np.abs(M).max()
    if peak < 1e-12: return
    H, W = M.shape; Y, X = np.mgrid[0:H, 0:W]
    lv = np.array([0.25, 0.5, 0.75]) * peak
    ax.contour(X, Y, M, levels=lv, colors='#8b1a1a', linewidths=0.8, zorder=4)
    ax.contour(X, Y, M, levels=-lv[::-1], colors='#12506b', linewidths=0.8, zorder=4)

def despine(ax):
    for s in ('top', 'right'): ax.spines[s].set_visible(False)

In [ ]:
fig = plt.figure(figsize=(fs.W_FULL, fs.W_FULL * 0.42))
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.0, 1.25], wspace=0.34,
                      left=0.06, right=0.98, top=0.80, bottom=0.15)

axA = fig.add_subplot(gs[0, 0]); despine(axA)
stages = ['reconstructed', 'pass QC\n($R^2\\geq0.3$)', 'verified\nsimple ($m{=}1$)']
counts = [3302, N_POP, 31]
ypos = np.arange(len(stages))[::-1]
for y, c, lab in zip(ypos, counts, stages):
    w = np.log10(c)
    axA.barh(y, w, height=0.55, color=fs.GREY, edgecolor=fs.INK, linewidth=0.6)
    axA.text(w + 0.06, y, f'{c:,}', va='center', fontsize=fs.FS_ANNOT,
             fontweight='bold')
    axA.text(-0.05, y, lab, va='center', ha='right', fontsize=fs.FS_ANNOT)
axA.set_xlim(0, 4.4); axA.set_ylim(-0.6, len(stages)-0.4)
axA.set_yticks([]); axA.set_xticks([])
axA.spines['left'].set_visible(False); axA.spines['bottom'].set_visible(False)
axA.set_title('Attrition to the analysed population', fontsize=fs.FS_ANNOT,
              loc='left', pad=10)
fs.panel_label(axA, 'A', dx=-0, dy=1.275)

axB = fig.add_subplot(gs[0, 1]); despine(axB)
auto = [ (POP['derivative_order']==k).sum() for k in (0,1,2) ]
fin  = [ (POP['m_final']==k).sum() for k in (0,1,2) ]
x = np.arange(3); wdt = 0.38
axB.bar(x - wdt/2, auto, wdt, label='automated',
        color=[fs.ORDER[k] for k in (0,1,2)], alpha=0.55, edgecolor=fs.INK, lw=0.5)
axB.bar(x + wdt/2, fin, wdt, label='verified',
        color=[fs.ORDER[k] for k in (0,1,2)], edgecolor=fs.INK, lw=0.5)
ymax = max(max(auto), max(fin))
for xi, (a, f_) in enumerate(zip(auto, fin)):
    axB.text(xi - wdt/2, a + ymax*0.015, str(a), ha='center', fontsize=8)
    axB.text(xi + wdt/2, f_ + ymax*0.015, str(f_), ha='center', fontsize=8)
axB.set_xticks(x); axB.set_xticklabels([fs.ORDER_LABEL[k] for k in (0,1,2)])
axB.set_ylabel('neurons'); axB.set_ylim(0, ymax*1.12)
axB.legend(frameon=False, fontsize=8, loc='upper right')
axB.set_title('Derivative-order composition', fontsize=fs.FS_ANNOT,
              loc='left', pad=10)
fs.panel_label(axB, 'B', dx=0, dy=1.275)

axC = fig.add_subplot(gs[0, 2])
best = max(dataset, key=lambda r: r['r_squared'])
signed_map_axes(axC, best['rf_smooth'],
                f"representative $m{{=}}1$ cell\n({best['name']}, $R^2={best['r_squared']:.2f}$)")
axC.title.set_position((0.0, 0.96)); axC.title.set_ha('left')
model_contours(axC, np.asarray(best['model_map'], float))
fs.panel_label(axC, 'C', dx=0, dy=1.275)

fs.save(fig, 'fig02_funnel_order', outdir=FIG_DIR)
plt.show()

In [ ]:
order = sorted(dataset, key=lambda r: r['kappa_dir'])
ncol, nrow = 6, int(np.ceil(len(order) / 6))

row_h = fs.W_FULL / ncol * (9/16) * 1.55
fig_h = row_h * nrow + 0.55
fig = plt.figure(figsize=(fs.W_FULL, fig_h))

title_frac = 0.75 / fig_h
fig.text(0.02, 1 - title_frac * 0.45,
         'The 31 verified first-order simple cells, ordered by elongation '
         r'$\kappa_{\mathrm{dir}}$', fontsize=11, ha='left', va='center')

gs = fig.add_gridspec(nrow, ncol, wspace=0.12, hspace=0.66,
                      left=0.02, right=0.98,
                      top=1 - title_frac, bottom=0.015)

for i, rec in enumerate(order):
    r, c = divmod(i, ncol)
    ax = fig.add_subplot(gs[r, c])
    signed_map_axes(ax, rec['rf_smooth'])
    model_contours(ax, np.asarray(rec['model_map'], float))
    ax.set_title(f"{rec['name']}\n$\\kappa_{{\\mathrm{{dir}}}}={rec['kappa_dir']:.2f}$",
                 fontsize=7.4, pad=2.5, linespacing=1.05)
fs.save(fig, 'fig03_gallery', outdir=FIG_DIR)
plt.show()

In [ ]:
fig = plt.figure(figsize=(fs.W_FULL, fs.W_FULL * 0.34))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], wspace=0.34,
                      left=0.06, right=0.97, top=0.84, bottom=0.20)

axA = fig.add_subplot(gs[0, 0]); despine(axA)
axA.hist(m1['sigma_deg'], bins=np.arange(4, 14, 1.0), color=fs.GREY,
         edgecolor=fs.INK, linewidth=0.6)
axA.axvline(m1['sigma_deg'].median(), color=fs.RUST, lw=1.6)
axA.text(m1['sigma_deg'].median()+0.2, axA.get_ylim()[1]*0.9,
         f"median\n{m1['sigma_deg'].median():.1f}$^\\circ$",
         color=fs.RUST, fontsize=8, va='top')
axA.set_xlabel(r'geom.-mean scale $\sigma$ (deg)')
axA.set_ylabel('cells')
fs.panel_label(axA, 'A', dx=-0.22, dy=1.10)

axB = fig.add_subplot(gs[0, 1]); despine(axB)
kd = m1['kappa_dir'].values
kl = np.array([r.get('kappa_lobe', np.nan) for r in dataset])
bins = np.arange(0.3, 2.4, 0.2)
axB.hist(kd, bins=bins, color=fs.RUST, alpha=0.55, edgecolor=fs.INK,
         linewidth=0.5, label=r'$\kappa_{\mathrm{dir}}$')
kl_finite = kl[np.isfinite(kl)]
if len(kl_finite):
    axB.hist(kl_finite, bins=bins, color=fs.PETROL, alpha=0.45,
             edgecolor=fs.INK, linewidth=0.5, label=r'$\kappa_{\mathrm{lobe}}$ (control)')
axB.axvline(1.0, color=fs.GREY, lw=0.8, ls='--')
axB.set_xlabel('elongation'); axB.set_ylabel('cells')
axB.legend(frameon=False, fontsize=8)
fs.panel_label(axB, 'B', dx=-0.22, dy=1.10)

axC = fig.add_subplot(gs[0, 2], projection='polar')
for ang, col, lab in [(m1['phi_deg'], fs.RUST, r'$\varphi$ (lobe)'),
                      (m1['theta_envelope'], fs.PETROL, r'$\theta$ (envelope)')]:
    a = np.deg2rad(ang.astype(float) % 360)
    axC.plot(a, np.ones_like(a) + (0.04 if col is fs.RUST else -0.04),
             'o', color=col, ms=4, alpha=0.8, label=lab)
axC.set_rticks([]); axC.set_ylim(0, 1.25)
axC.set_theta_zero_location('E'); axC.set_theta_direction(1)
axC.set_xticks(np.deg2rad([0,90,180,270]))
axC.set_xticklabels(['0°','90°','180°','270°'], fontsize=8)
axC.legend(frameon=False, fontsize=7.5, loc='lower center',
           bbox_to_anchor=(0.5, -0.42))
axC.set_title('orientation', fontsize=fs.FS_ANNOT, pad=10)
fs.panel_label(axC, 'C', dx=-0.10, dy=1.12)

fs.save(fig, 'fig04_spatial_params', outdir=FIG_DIR)
plt.show()

In [ ]:
fig = plt.figure(figsize=(fs.W_FULL, fs.W_FULL * 0.34))
gs = fig.add_gridspec(1, 3, wspace=0.42, left=0.07, right=0.98,
                      top=0.82, bottom=0.22)

SIGMA_SMOOTH = 0.75 * PIXEL_SIZE_DEG
if 'sigma_minor' not in POP.columns:
    POP['sigma_minor'] = POP[['sigma_x', 'sigma_y']].min(axis=1)

axA = fig.add_subplot(gs[0, 0]); despine(axA)

pop = POP.dropna(subset=['sigma', 'kappa'])
r_p, _ = pearsonr(pop['sigma'], pop['kappa'])

axA.scatter(pop['sigma'], pop['kappa'], s=5, c=fs.GREY_MID,
            alpha=0.30, edgecolor='none', label=f'population ($n={N_POP:,}$)')
axA.scatter(m1['sigma_deg'], m1['kappa'], s=24, c=fs.RUST,
            edgecolor='white', linewidth=0.4, label='verified $m{=}1$', zorder=4)

axA.set_ylim(1.0, 3)

axA.set_xlabel(r'$\sigma$ (deg)')
axA.set_ylabel(r'$\kappa\geq1$')
axA.set_title(f'$r={r_p:.2f}$', fontsize=fs.FS_ANNOT, loc='left', pad=6)
axA.legend(frameon=False, fontsize=7.5, loc='upper right')
fs.panel_label(axA, 'A', dx=-0.26, dy=1.14)

axB = fig.add_subplot(gs[0, 1]); despine(axB)
axB.scatter(m1['kappa_dir'], m1['sigma_deg'], s=28, c=fs.RUST,
            edgecolor='white', linewidth=0.4)
axB.axvline(1.0, color=fs.GREY, lw=0.8, ls='--')
rb, pb = pearsonr(m1['kappa_dir'], m1['sigma_deg'])
axB.set_xlabel(r'$\kappa_{\mathrm{dir}}$')
axB.set_ylabel(r'geom.-mean $\sigma$ (deg)')
axB.set_title(f'$r={rb:.2f}$, $p={pb:.2f}$', fontsize=fs.FS_ANNOT,
              loc='left', pad=6)
fs.panel_label(axB, 'B', dx=-0.30, dy=1.14)

axC = fig.add_subplot(gs[0, 2]); despine(axC)
axC.scatter(m1['kappa_dir'], m1['sigma_min'], s=28, c=fs.PETROL,
            edgecolor='white', linewidth=0.4)
axC.axhline(SIGMA_SMOOTH, color=fs.GREY, lw=0.8, ls='--')
axC.text(m1['kappa_dir'].max(), SIGMA_SMOOTH + 0.1,
         r'$\sigma_{\mathrm{smooth}}$', color=fs.GREY, fontsize=7.5, ha='right')
axC.axvline(1.0, color=fs.GREY, lw=0.8, ls='--')
rc, pc = pearsonr(m1['kappa_dir'], m1['sigma_min'])
axC.set_xlabel(r'$\kappa_{\mathrm{dir}}$')
axC.set_ylabel(r'$\min(\sigma_\varphi,\sigma_\perp)$ (deg)')
axC.set_title(f'$r={rc:.2f}$, $p={pc:.2f}$', fontsize=fs.FS_ANNOT,
              loc='left', pad=6)
fs.panel_label(axC, 'C', dx=-0.30, dy=1.14)

fs.save(fig, 'fig05_scale_elongation', outdir=FIG_DIR)
plt.show()

In [ ]:
fig = plt.figure(figsize=(fs.W_FULL, fs.W_FULL * 0.34))
gs = fig.add_gridspec(1, 3, wspace=0.75, left=0.07, right=0.97,
                      top=0.82, bottom=0.22)

axA = fig.add_subplot(gs[0, 0]); despine(axA)
sc = axA.scatter(m1['cortex_x_um'], m1['cortex_y_um'], c=m1['kappa_dir'],
                 cmap='PuOr_r', s=42, edgecolor=fs.INK, linewidth=0.4,
                 vmin=0.4, vmax=1.8)
cb = fig.colorbar(sc, ax=axA, fraction=0.046, pad=0.14)
cb.set_label(r'$\kappa_{\mathrm{dir}}$', fontsize=8, labelpad=1); cb.ax.tick_params(labelsize=7)
axA.set_xlabel('cortical x (µm)'); axA.set_ylabel('cortical y (µm)')
fs.panel_label(axA, 'A', dx=-0.30, dy=1.14)
axA.set_title('RF centres in cortex', fontsize=fs.FS_ANNOT, loc='left', pad=8)

axB = fig.add_subplot(gs[0, 1]); despine(axB)
dist, dk = [], []
for cid, sub in POP.dropna(subset=['cortex_x_um','cortex_y_um','kappa']).groupby('container_id'):
    if len(sub) < 10: continue
    D = squareform(pdist(sub[['cortex_x_um','cortex_y_um']].values))
    K = squareform(pdist(sub['kappa'].values[:, None]))
    idx = np.triu_indices(len(sub), 1)
    dist.extend(D[idx]); dk.extend(K[idx])
dist, dk = np.array(dist), np.array(dk)
edges = np.linspace(0, np.percentile(dist, 95), 12)
mid = 0.5*(edges[:-1]+edges[1:])
means = [dk[(dist>=lo)&(dist<hi)].mean() for lo, hi in zip(edges[:-1], edges[1:])]
axB.plot(mid, means, '-o', color=fs.PETROL, ms=4, lw=1.4)
r_dk, p_dk = pearsonr(dist, dk)
axB.set_xlabel('pairwise cortical distance (µm)')
axB.set_ylabel(r'mean $|\Delta\kappa|$')
axB.set_title(f'$r={r_dk:.3f}$, $p={p_dk:.2f}$', fontsize=fs.FS_ANNOT,
              loc='left', pad=6)
fs.panel_label(axB, 'B', dx=-0.32, dy=1.14)

axC = fig.add_subplot(gs[0, 2]); despine(axC)
m1 = m1.copy()
m1['zone'] = np.where(m1['x0_deg'] < 40, 'binoc', 'monoc')
data = [m1.loc[m1['zone']=='binoc','sigma_deg'], m1.loc[m1['zone']=='monoc','sigma_deg'],
        m1.loc[m1['zone']=='binoc','kappa_dir'], m1.loc[m1['zone']=='monoc','kappa_dir']]
positions = [1, 2, 4, 5]
cols = [fs.PETROL, fs.GREY, fs.PETROL, fs.GREY]
bp = axC.boxplot(data, positions=positions, widths=0.7, patch_artist=True,
                 showfliers=False)
for patch, col in zip(bp['boxes'], cols):
    patch.set_facecolor(col); patch.set_alpha(0.55); patch.set_edgecolor(fs.INK)
for med in bp['medians']: med.set_color(fs.INK)
axC.set_xticks([1.5, 4.5]); axC.set_xticklabels([r'$\sigma$ (deg)', r'$\kappa_{\mathrm{dir}}$'])
from matplotlib.patches import Patch
axC.legend([Patch(fc=fs.PETROL, alpha=0.55), Patch(fc=fs.GREY, alpha=0.55)],
           ['binocular', 'monocular'], frameon=False, fontsize=7.5, loc='upper right')
fs.panel_label(axC, 'C', dx=-0.24, dy=1.14)

fs.save(fig, 'fig06_organisation', outdir=FIG_DIR)
plt.show()

In [ ]:
gab = pd.read_csv(GABOR_CSV)

fig = plt.figure(figsize=(fs.W_THREEQ, fs.W_THREEQ * 0.50))
gs = fig.add_gridspec(1, 2, wspace=0.34, left=0.09, right=0.97,
                      top=0.82, bottom=0.17)

n = len(gab)
n_gab = int((gab['r2_gabor'] > gab['r2_gd']).sum())

axA = fig.add_subplot(gs[0, 0]); despine(axA)
lims = [min(gab['r2_gd'].min(), gab['r2_gabor'].min())-0.03,
        max(gab['r2_gd'].max(), gab['r2_gabor'].max())+0.03]
axA.scatter(gab['r2_gd'], gab['r2_gabor'], s=26, c=fs.PETROL,
            edgecolor='white', linewidth=0.4, zorder=3)
axA.plot(lims, lims, '--', color=fs.GREY, lw=1)
axA.set(xlim=lims, ylim=lims,
        xlabel=r'$R^2$ Gaussian derivative ($k{=}7$)',
        ylabel=r'$R^2$ Gabor ($k{=}9$)')
axA.set_title(f'Gabor ahead in {n_gab}/{n}', fontsize=fs.FS_ANNOT,
              loc='left', pad=3)
fs.panel_label(axA, 'A', dx=-0.20, dy=1.10)

axB = fig.add_subplot(gs[0, 1]); despine(axB)
if 'kappa_dir_gd' in gab and 'kappa_dir_gabor' in gab:
    axB.scatter(gab['kappa_dir_gd'], gab['kappa_dir_gabor'], s=26, c=fs.RUST,
                edgecolor='white', linewidth=0.4, zorder=3)
    kl = [0.3, max(gab['kappa_dir_gd'].max(), gab['kappa_dir_gabor'].max())*1.05]
    axB.plot(kl, kl, '--', color=fs.GREY, lw=1)
    axB.axhline(1, color=fs.GREY_MID, lw=0.7); axB.axvline(1, color=fs.GREY_MID, lw=0.7)
    axB.set(xlim=kl, ylim=kl,
            xlabel=r'$\kappa_{\mathrm{dir}}$ Gaussian derivative',
            ylabel=r'$\kappa_{\mathrm{dir}}$ Gabor')
    axB.set_title('shape estimate is model-dependent', fontsize=fs.FS_ANNOT,
                  loc='left', pad=3)
else:
    axB.text(0.5, 0.5, 'kappa_dir columns not in CSV\n(rerun nb11 v3)',
             ha='center', va='center', transform=axB.transAxes, fontsize=8)
fs.panel_label(axB, 'B', dx=-0.20, dy=1.10)

fs.save(fig, 'fig07_gabor', outdir=FIG_DIR)
plt.show()

In [ ]:
for stem in ['fig02_funnel_order','fig03_gallery','fig04_spatial_params',
             'fig05_scale_elongation','fig06_organisation','fig07_gabor']:
    p = FIG_DIR / f'{stem}.pdf'

In [ ]:
def rayleigh_test(angles_rad):
    """Rayleigh test for non-uniformity of circular data. Returns (R, p, n)."""
    n = len(angles_rad)
    C = np.cos(angles_rad).sum(); S = np.sin(angles_rad).sum()
    R = np.hypot(C, S) / n
    z = n * R**2
    p = np.exp(-z) * (1 + (2*z - z**2)/(4*n)
                        - (24*z - 132*z**2 + 76*z**3 - 9*z**4)/(288*n**2))
    return R, float(np.clip(p, 0, 1)), n

phi = pd.to_numeric(m1['phi_deg'], errors='coerce').dropna().values
if phi.max() <= 180:
    raise ValueError('phi values appear to be in [0,180) — use the Sep-13 pickle with the [0,360) fix.')

R_dir, p_dir, n = rayleigh_test(np.deg2rad(phi % 360))
R_ax,  p_ax,  _ = rayleigh_test(np.deg2rad((phi % 180) * 2))

